# `06-deepagent/test.ipynb`
1. csv 파일을 받아서
2. 데이터 분석을 진행하고
3. 슬랙으로 결과를 보냄

In [13]:
from dotenv import load_dotenv

load_dotenv()

True

In [14]:
from daytona import Daytona
from langchain_daytona import DaytonaSandbox

sandbox = Daytona().create()
backend = DaytonaSandbox(sandbox=sandbox)

In [15]:
result = backend.execute('ls -a')

In [16]:
print(result.output)

.
..
.bash_logout
.bashrc
.daytona
.face
.face.icon
.profile
.zshrc


In [17]:
import csv
import io

data = [
    ["Date", "Product", "Units Sold", "Revenue"],
    ["2025-08-01", "Widget A", 10, 250],
    ["2025-08-02", "Widget B", 5, 125],
    ["2025-08-03", "Widget A", 7, 175],
    ["2025-08-04", "Widget C", 3, 90],
    ["2025-08-05", "Widget B", 8, 200],
]

buf = io.StringIO()
writer = csv.writer(buf)
writer.writerows(data)

# 원본 csv 데이터
csv_bytes = buf.getvalue().encode('utf-8')
buf.close()

# 파일 저장(지금 우리는 필요 없음)
with open('./sales.csv', 'wb') as f:
    f.write(csv_bytes)

# sandbox 에 업로드
backend.upload_files(
    [('/home/daytona/data/sales_data.csv', csv_bytes)]
)

[FileUploadResponse(path='/home/daytona/data/sales_data.csv', error=None)]

### Slack
- Agent가 사용할 Slack Messaging Tool 생성

In [18]:
import os
from langchain.tools import tool
from slack_sdk import WebClient

SLACK_BOT_TOKEN = os.getenv('SLACK_BOT_TOKEN')

client = WebClient(token=SLACK_BOT_TOKEN)

# 봇이 접근 가능한 모든 채널 리스트
res = client.conversations_list()

res['channels']

for ch in res['channels']:
    print(ch['name'], ch['id'])

social_channel_id = 'C0A3ZF1GR7F'

client.chat_postMessage(
    channel=social_channel_id,
    text='hello'
)

새-채널 C0A3MF1MNP9
새-워크스페이스-전체 C0A3WGG1R7V
소셜 C0A3ZF1GR7F


In [20]:
import os
from langchain.tools import tool
from slack_sdk import WebClient

SLACK_TOKEN = os.getenv('SLACK_BOT_TOKEN')
slack_client = WebClient(token=SLACK_TOKEN)

@tool(parse_docstring=True)
def send_slack_message(text: str, file_path: str | None = None) -> str:
    """메세지를 전송하고, 필요한 경우 이미지 등 파일을 첨부합니다.

    Args:
        text: 전송할 메세지의 내용입니다.
        file_path: 파일 시스템에 저장된 첨부 파일의 경로입니다. (선택 사항)
    """
    social_channel_id = 'C0A3ZF1GR7F'
    # ... 나머지 로직 동일
    # 첨부 파일 없으면
    if not file_path:
        slack_client.chat_postMessage(channel=social_channel_id, text=text)
    else:
        fp = backend.download_files([file_path])
        slack_client.file_upload_v2(
            channel=social_channel_id,
            content=fp[0].content,
            initial_comment=text
        )
    return 'Message sent'

In [21]:
import uuid

from langgraph.checkpoint.memory import InMemorySaver
from deepagents import create_deep_agent

checkpointer = InMemorySaver()

agent = create_deep_agent(
    model='openai:gpt-5-mini',
    tools=[send_slack_message],
    backend=backend,
    checkpointer=checkpointer
)

thread_id = str(uuid.uuid4())
config = {'configurable': {'thread_id': thread_id}}

In [ ]:
input_message = {
    'role': 'user',
    'content': '''
    현재 폴더 안에 ./data/sales_data.csv 파일을 분석하고, 시각화 해줘.
    다 끝나면 분석결과와 시각화 이미지를 Slack 메세지로 보내줘.
    '''
}

for step in agent.stream(
    {"messages": [input_message]},
    config,
    stream_mode="updates",
):
    for _, update in step.items():
        if update and (messages := update.get("messages")) and isinstance(messages, list):
            for message in messages:
                message.pretty_print()

================================== Ai Message ==================================

[{'id': 'rs_0f5eacd8af5558590069afa79d83e48197862047da61cfb4b1', 'summary': [], 'type': 'reasoning'}, {'arguments': '{"todos":[{"content":"Locate ./data/sales_data.csv in repository and preview it","status":"in_progress"},{"content":"Run analysis script to produce summary and visualizations","status":"pending"},{"content":"Send analysis summary and visualization image to Slack","status":"pending"},{"content":"Report completion to user","status":"pending"}]}', 'call_id': 'call_qNQ2qDVEwzyAcYcbVn0muQyd', 'name': 'write_todos', 'type': 'function_call', 'id': 'fc_0f5eacd8af5558590069afa7b8ff648197a6f2afb370b4b57e', 'status': 'completed'}]
Tool Calls:
  write_todos (call_qNQ2qDVEwzyAcYcbVn0muQyd)
 Call ID: call_qNQ2qDVEwzyAcYcbVn0muQyd
  Args:
    todos: [{'content': 'Locate ./data/sales_data.csv in repository and preview it', 'status': 'in_progress'}, {'content': 'Run analysis script to produce summary and vi